# BiRefNet-portrait — Fine-tuning on Jackets Dataset
**Before running:** Runtime → Change runtime type → T4 GPU

Run cells top to bottom. Cell 6 will ask for your HuggingFace token.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers timm einops kornia

In [ ]:
# Cell 3 ? Extract dataset
# UPDATE this path if you placed the zip in a subfolder in Drive
DRIVE_ZIP = '/content/drive/MyDrive/jackets_dataset.zip'

import zipfile, os
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall('/content/data')

IM_DIR = '/content/data/jackets_dataset/im'
GT_DIR = '/content/data/jackets_dataset/gt'

# Images held out from training ? used only for validation in Cell 9
VALIDATION_NAMES = {'suit_043_front', 'suit_043_back', 'suit_044_back', 'suit_048_back'}

# If this check fails the zip still has Windows backslash paths; re-upload the fixed zip
if not os.path.isdir(IM_DIR):
    print('ERROR: IM_DIR not found. Extracted structure:')
    for root, dirs, files in os.walk('/content/data'):
        if files:
            print(f'  {root}: {files[:3]}')
    raise FileNotFoundError(f'Missing: {IM_DIR}')

im_files = sorted(os.listdir(IM_DIR))
gt_files = sorted(os.listdir(GT_DIR))
train_count = sum(1 for f in im_files if os.path.splitext(f)[0] not in VALIDATION_NAMES)
print(f'Images: {len(im_files)}, Masks: {len(gt_files)}')
print(f'Training: {train_count}, Validation held out: {len(VALIDATION_NAMES)}')
print(f'Sample files: {im_files[:3]}')
assert len(im_files) == len(gt_files)
assert train_count >= 10


In [ ]:
# Cell 4 — Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU — change runtime to T4'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 5 — Dataset, augmentation, loss
import random
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 512 instead of 1024 — required to fit in T4 VRAM alongside the model
SIZE = 512

class JacketsDataset(Dataset):
    def __init__(self, im_dir, gt_dir, augment=True, exclude=None):
        self.im_dir  = Path(im_dir)
        self.gt_dir  = Path(gt_dir)
        self.augment = augment
        exclude = exclude or set()
        # Exclude validation images so the model never sees them during training
        self.names = sorted(
            p.stem for p in self.im_dir.glob('*.jpg') if p.stem not in exclude
        )

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img  = Image.open(self.im_dir / f'{name}.jpg').convert('RGB')
        mask = Image.open(self.gt_dir / f'{name}.png').convert('L')

        if self.augment:
            if random.random() > 0.5:
                img  = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.6:
                angle = random.uniform(-8, 8)
                img  = img.rotate(angle, fillcolor=(255, 255, 255))
                mask = mask.rotate(angle, fillcolor=0)
            img = transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.1
            )(img)

        img  = img.resize((SIZE, SIZE), Image.LANCZOS)
        mask = mask.resize((SIZE, SIZE), Image.NEAREST)

        img_t  = transforms.Normalize(
            [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        )(transforms.ToTensor()(img))
        mask_t = torch.tensor(
            np.array(mask) / 255.0, dtype=torch.float32
        ).unsqueeze(0)

        return img_t, mask_t


def bce_iou_loss(pred, target):
    # Resize target to match prediction scale (multi-scale supervision)
    if pred.shape[-2:] != target.shape[-2:]:
        target = F.interpolate(target, size=pred.shape[-2:], mode='nearest')
    # bce_with_logits is autocast-safe (combines sigmoid internally)
    bce = F.binary_cross_entropy_with_logits(pred, target)
    p   = pred.float().sigmoid()
    t   = target.float()
    inter = (p * t).sum(dim=(2, 3))
    union = (p + t - p * t).sum(dim=(2, 3))
    iou   = 1 - (inter / (union + 1e-8)).mean()
    return bce + iou




dataset = JacketsDataset(IM_DIR, GT_DIR, augment=True, exclude=VALIDATION_NAMES)
assert len(dataset) > 0, f'Dataset empty ? no .jpg files found in {IM_DIR}. Run Cell 3 first.'
loader  = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
print(f'Training set: {len(dataset)} images at {SIZE}x{SIZE}')

In [ ]:
# Cell 6 — Load BiRefNet-portrait
from getpass import getpass
from transformers import AutoModelForImageSegmentation

HF_TOKEN = getpass('HuggingFace READ token: ')  # paste token, won't be shown

model = AutoModelForImageSegmentation.from_pretrained(
    'ZhengPeng7/BiRefNet-portrait',
    trust_remote_code=True,
    token=HF_TOKEN
).cuda()

print('Model loaded on', next(model.parameters()).device)

In [ ]:
# Cell 7 — Training
import torch.nn as nn

EPOCHS    = 30
LR        = 1e-5
ACCUM     = 4
SAVE_DIR  = '/content/birefnet_jackets_finetuned'
DRIVE_OUT = '/content/drive/MyDrive/birefnet_jackets_finetuned'

# Freeze backbone ? preserve edge detection, only train decoder
for name, param in model.named_parameters():
    if name.startswith('bb.'):
        param.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen   = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Trainable: {trainable:,} | Frozen: {frozen:,}')

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda')
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    # BatchNorm must stay in eval mode with batch_size=1 to avoid size=[1,C,1,1] error
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(loader):
        imgs, masks = imgs.cuda(), masks.cuda()

        with torch.amp.autocast('cuda'):
            # Model returns: [tuple([list, list]), list([None])]
            # Predictions at 4 scales live in model_output[0][1]
            pred_tensors = model(imgs)[0][1]
            loss = sum(bce_iou_loss(p, masks) for p in pred_tensors) / len(pred_tensors)

        scaler.scale(loss / ACCUM).backward()

        if (step + 1) % ACCUM == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item()

    scheduler.step()
    avg = epoch_loss / len(loader)
    tag = ''
    if avg < best_loss:
        best_loss = avg
        model.save_pretrained(SAVE_DIR)
        tag = '  ← best, saved'
    print(f'Epoch {epoch+1:02d}/{EPOCHS}  loss={avg:.4f}{tag}')

print(f'\nDone. Best loss: {best_loss:.4f}')

# Auto-save to Drive immediately after training — prevents loss if Colab disconnects
import shutil, os
if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(SAVE_DIR, DRIVE_OUT)
print(f'Model saved to Drive: {DRIVE_OUT}')

In [ ]:
# Cell 8 — Save best model to Google Drive
DRIVE_OUT = '/content/drive/MyDrive/birefnet_jackets_finetuned'
!cp -r {SAVE_DIR} {DRIVE_OUT}
print(f'Saved to Drive: {DRIVE_OUT}')

In [ ]:
# Cell 9 — Visual validation: before vs after on held-out images
# These are images the model never saw during training
import matplotlib.pyplot as plt
from torchvision import transforms as T

model_orig = AutoModelForImageSegmentation.from_pretrained(
    'ZhengPeng7/BiRefNet-portrait', trust_remote_code=True, token=HF_TOKEN
).cuda().eval()

model.eval()

tf = T.Compose([
    T.Resize((SIZE, SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_names = ['suit_043_front', 'suit_043_back', 'suit_044_back', 'suit_048_back']

fig, axes = plt.subplots(len(test_names), 4, figsize=(16, 4 * len(test_names)))

for row, name in enumerate(test_names):
    img = Image.open(f'{IM_DIR}/{name}.jpg').convert('RGB')
    gt  = Image.open(f'{GT_DIR}/{name}.png').convert('L')
    inp = tf(img).unsqueeze(0).cuda()

    with torch.no_grad():
        # model output[0][1][-1] = final-scale prediction tensor
        pred_orig  = model_orig(inp)[0][1][-1].float().sigmoid().squeeze().cpu().numpy()
        pred_tuned = model(inp)[0][1][-1].float().sigmoid().squeeze().cpu().numpy()

    axes[row][0].imshow(img.resize((512, 512)))
    axes[row][0].set_title(f'Input: {name}')
    axes[row][1].imshow(np.array(gt.resize((512, 512))), cmap='gray')
    axes[row][1].set_title('Ground truth mask')
    axes[row][2].imshow(pred_orig,  cmap='gray')
    axes[row][2].set_title('BiRefNet-portrait (before)')
    axes[row][3].imshow(pred_tuned, cmap='gray')
    axes[row][3].set_title('Fine-tuned (after)')
    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/jackets_finetune_results.png', dpi=80)
plt.show()
print('Results saved to Drive: jackets_finetune_results.png')